# 历史 Token 套餐盈利测算（已停用）

## tl;dr

> 本 Notebook 使用已废弃的 ¥1/850,000 Token、¥6/1,700,000 Token 假设，不再代表当前产品。现行 ¥1/10、¥6/30、¥12/75 狗头模型见 `analysis/current-pricing-unit-economics.md`。以下内容仅保留为历史敏感性分析。

## Context & Methods

### Key Assumptions

- 核验日期：2026-07-21。
- 阶跃 `step-3.5-flash`：未缓存输入 0.70 元/百万、缓存输入 0.14 元/百万、输出 2.10 元/百万。
- 一次标准回复：6,000 未缓存输入 Token + 1,000 输出 Token，模型成本 0.0063 元，对应 9,000 个狗头军师计费 Token。
- 新客包：1 元/850,000 Token；标准包：6 元/1,700,000 Token。
- 设备支付结构假设：70% Android/鸿蒙/Windows，30% iOS；活动费率分别为 1% 和 12%，加权费率 4.3%。
- 所有已售 Token 最终全部使用，因而完整计提未来模型成本；这比只看充值现金更保守。
- 未计服务器、内容安全、退款、税、投放和开发者人工。

In [ ]:
import pandas as pd

input_tokens = 6_000
output_tokens = 1_000
input_price_per_million = 0.70
output_price_per_million = 2.10
billable_tokens_per_reply = input_tokens + output_tokens * 3
model_cost_per_reply = (input_tokens * input_price_per_million + output_tokens * output_price_per_million) / 1_000_000
weighted_payment_rate = 0.70 * 0.01 + 0.30 * 0.12

packs = pd.DataFrame([
    {"pack": "1元新客包", "price": 1.0, "billable_tokens": 850_000, "model_cost": 0.595},
    {"pack": "6元标准包", "price": 6.0, "billable_tokens": 1_700_000, "model_cost": 1.190},
])
packs["theoretical_replies"] = packs["billable_tokens"] / billable_tokens_per_reply
packs["net_cash_after_payment"] = packs["price"] * (1 - weighted_payment_rate)
packs["contribution_profit"] = packs["net_cash_after_payment"] - packs["model_cost"]
packs["contribution_per_reply"] = packs["contribution_profit"] / packs["theoretical_replies"]
packs.round(4)

## Results

成熟期以 6 元标准包为主。下面三档不是用户预测，而是用于回答“达到某个月活规模时，大约能赚多少”的敏感性测算。

In [ ]:
standard = packs.loc[packs["pack"] == "6元标准包"].iloc[0]
scenarios = pd.DataFrame([
    {"scenario": "保守", "paid_rate": 0.02, "monthly_replies_per_payer": 10},
    {"scenario": "正常", "paid_rate": 0.05, "monthly_replies_per_payer": 30},
    {"scenario": "乐观", "paid_rate": 0.10, "monthly_replies_per_payer": 60},
])
rows = []
for mau in [1_000, 10_000, 100_000]:
    for _, s in scenarios.iterrows():
        payers = mau * s.paid_rate
        monthly_replies = payers * s.monthly_replies_per_payer
        revenue = monthly_replies / standard.theoretical_replies * standard.price
        contribution = monthly_replies * standard.contribution_per_reply
        rows.append({
            "MAU": mau,
            "scenario": s.scenario,
            "paid_users": int(payers),
            "token_revenue": revenue,
            "token_contribution_profit": contribution,
        })
scenario_results = pd.DataFrame(rows)
scenario_results.round(2)

广告无法在没有真实流量主数据时预测。下面只给 1 万 MAU、95% 免费用户、每人每月 10 次回复、每 5 次回复最多一次广告、70% 有效填充时的 eCPM 敏感性。

In [ ]:
mau_for_ads = 10_000
free_user_share = 0.95
replies_per_free_user = 10
replies_per_ad_opportunity = 5
fill_rate = 0.70
effective_impressions = mau_for_ads * free_user_share * replies_per_free_user / replies_per_ad_opportunity * fill_rate
ad_sensitivity = pd.DataFrame({
    "eCPM": [5, 15, 30],
})
ad_sensitivity["monthly_ad_revenue"] = effective_impressions / 1_000 * ad_sensitivity["eCPM"]
free_user_model_cost = mau_for_ads * free_user_share * replies_per_free_user * model_cost_per_reply
ad_sensitivity["free_user_model_cost"] = free_user_model_cost
ad_sensitivity["ad_net_contribution"] = ad_sensitivity["monthly_ad_revenue"] - free_user_model_cost
break_even_ad_ecpm = free_user_model_cost / effective_impressions * 1_000
ad_sensitivity.round(2)

In [ ]:
per_mau = scenarios.copy()
per_mau["contribution_per_MAU"] = per_mau["paid_rate"] * per_mau["monthly_replies_per_payer"] * standard.contribution_per_reply
per_mau["MAU_for_10k_monthly_contribution"] = 10_000 / per_mau["contribution_per_MAU"]
per_mau[["scenario", "contribution_per_MAU", "MAU_for_10k_monthly_contribution"]].round(0)

## Takeaways

- 6 元标准包完全使用后的单包贡献利润约 4.55 元，但它可支持约 189 次标准回复；30 次/月的用户约 6.3 个月才需要再次购买。
- 正常档下，1 万 MAU、5% 付费率、每名付费用户每月 30 次回复，Token 月收入约 476 元、贡献利润约 362 元。
- 正常档仅靠 Token 要达到每月 1 万元贡献利润，需要约 27.7 万 MAU；乐观档也需要约 6.9 万 MAU。
- 广告示例中，免费用户模型成本约 598.5 元/月，eCPM 需要达到约 45 元才能覆盖；eCPM 为 5–30 元时仍然亏损。
- 1 元包可支持约 94 次标准回复，可能把第一次 6 元复购推迟数月。高毛利率没有自动转化成高利润，真正约束是低绝对客单价和很慢的补充购买频率。

In [ ]:
assert round(model_cost_per_reply, 4) == 0.0063
assert round(weighted_payment_rate, 3) == 0.043
assert abs(standard.contribution_profit - 4.552) < 1e-9
assert len(scenario_results) == 9
print("All calculation checks passed.")